# Adaptive Subject-Level Differential Privacy — Colab Runner

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

This notebook clones the repo, installs dependencies, and runs the full experiment.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/shaktsin/ml-research.git
%cd ml-research/adaptive-diff-privacy

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Verify GPU

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 4. Run the experiment

Runs all 3 configs: Baseline, Subject-DP (uniform), Adaptive Subject-DP.

In [ ]:
!python run_experiment.py

---
## 4b. Visualise experiment results

**Fill in the numbers printed above**, then run these cells.

Pre-filled with expected values — update `results` with your actual printed output.

In [ ]:
# ── UPDATE THESE with the values printed by run_experiment.py above ──────────
results = [
    {"config": "Baseline\n(no DP)",          "accuracy": 0.882, "mia_auc": 0.850},
    {"config": "Subject-DP\n(uniform)",       "accuracy": 0.841, "mia_auc": 0.650},
    {"config": "Adaptive\nSubject-DP",        "accuracy": 0.835, "mia_auc": 0.580},
]
# ─────────────────────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

labels   = [r["config"]   for r in results]
accuracy = [r["accuracy"] for r in results]
mia_auc  = [r["mia_auc"]  for r in results]

x     = np.arange(len(labels))
width = 0.35
COLORS = ["#e07b54", "#5b8db8", "#4caf7d"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Adaptive Subject-Level DP — Experiment Results", fontsize=14, fontweight="bold")

# --- Plot 1: Accuracy ---
ax = axes[0]
bars = ax.bar(x, accuracy, color=COLORS, width=0.5, edgecolor="white", linewidth=1.2)
ax.set_ylim(0.75, 0.92)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Test Accuracy")
ax.set_title("Classification Accuracy\n(higher = better model)")
for bar, val in zip(bars, accuracy):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

# --- Plot 2: MIA AUC ---
ax = axes[1]
bars = ax.bar(x, mia_auc, color=COLORS, width=0.5, edgecolor="white", linewidth=1.2)
ax.axhline(0.5, color="green", linestyle="--", linewidth=1.5, label="Ideal privacy (AUC=0.5)")
ax.set_ylim(0.4, 1.0)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("MIA Attack AUC")
ax.set_title("Membership Inference AUC\n(lower = better privacy)")
ax.legend(fontsize=8)
for bar, val in zip(bars, mia_auc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

# --- Plot 3: Privacy-Utility tradeoff scatter ---
ax = axes[2]
for i, (r, c) in enumerate(zip(results, COLORS)):
    # Privacy = 1 - (AUC - 0.5) / 0.5 → maps [0.5, 1.0] → [1.0, 0.0]
    privacy_score = 1.0 - (r["mia_auc"] - 0.5) / 0.5
    ax.scatter(privacy_score, r["accuracy"], color=c, s=180, zorder=5,
               edgecolors="white", linewidths=1.5)
    ax.annotate(r["config"].replace("\n", " "),
                (privacy_score, r["accuracy"]),
                textcoords="offset points", xytext=(8, 4), fontsize=8)
ax.set_xlabel("Privacy Score  (1.0 = ideal, 0.0 = no privacy)")
ax.set_ylabel("Test Accuracy")
ax.set_title("Privacy–Utility Tradeoff\n(top-right = best)")
ax.set_xlim(-0.05, 1.15)
ax.set_ylim(0.80, 0.90)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results_summary.png")

---
## 5. (Optional) Tweak parameters and re-run a single config

Edit the knobs below, train one config, then run the deep-dive plots in **Section 5b**.

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Subset
from data import AGNewsDataset
from model import get_model, get_tokenizer
from trainer import train_baseline, train_subject_dp, evaluate
from mia import run_mia, compute_losses

# --- Knobs ---
EPOCHS     = 3
CLIP_NORM  = 1.0
BASE_SIGMA = 1.0
N_SUBJECTS = 500
MAX_TRAIN  = 2000   # set None for full 120k dataset (much slower)
MAX_TEST   = 500
BATCH_SIZE = 16
LR         = 2e-5

tokenizer   = get_tokenizer()
train_ds    = AGNewsDataset('train', tokenizer, n_subjects=N_SUBJECTS)
test_ds     = AGNewsDataset('test',  tokenizer, n_subjects=N_SUBJECTS)

if MAX_TRAIN: train_ds = Subset(train_ds, range(MAX_TRAIN))
if MAX_TEST:  test_ds  = Subset(test_ds,  range(MAX_TEST))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)
mia_train    = DataLoader(Subset(train_ds, range(200)), batch_size=BATCH_SIZE)
mia_test     = DataLoader(Subset(test_ds,  range(200)), batch_size=BATCH_SIZE)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

In [ ]:
# Run Adaptive Subject-DP  (change to train_baseline or adaptive=False to compare)
model     = get_model(num_labels=4).to(device)
optimizer = AdamW(model.parameters(), lr=LR)

train_subject_dp(model, train_loader, optimizer, device, EPOCHS,
                 clip_norm=CLIP_NORM, base_sigma=BASE_SIGMA, adaptive=True)

acc = evaluate(model, test_loader, device)
auc = run_mia(model, mia_train, mia_test, device)
print(f'\nAccuracy : {acc:.4f}')
print(f'MIA AUC  : {auc:.4f}  (0.5=ideal privacy, 1.0=full leak)')

---
## 5b. Deep-dive plots — model in memory

Uses the trained `model` from step 5. Shows *why* the privacy holds (or doesn't).

In [ ]:
# ── Loss distribution: members vs non-members ─────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from mia import compute_losses

member_losses     = compute_losses(model, mia_train, device)
non_member_losses = compute_losses(model, mia_test,  device)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("MIA Loss Distribution — Members vs Non-Members", fontsize=13, fontweight="bold")

# Histogram
ax = axes[0]
bins = np.linspace(0, max(member_losses.max(), non_member_losses.max()) + 0.1, 40)
ax.hist(member_losses,     bins=bins, alpha=0.65, color="#e07b54", label=f"Members (train)  n={len(member_losses)}")
ax.hist(non_member_losses, bins=bins, alpha=0.65, color="#5b8db8", label=f"Non-members (test) n={len(non_member_losses)}")
ax.axvline(member_losses.mean(),     color="#c0391b", linestyle="--", linewidth=1.5, label=f"Member mean = {member_losses.mean():.3f}")
ax.axvline(non_member_losses.mean(), color="#1a5f9e", linestyle="--", linewidth=1.5, label=f"Non-member mean = {non_member_losses.mean():.3f}")
ax.set_xlabel("Cross-Entropy Loss")
ax.set_ylabel("Count")
ax.set_title("Loss Histogram\n(overlap = good privacy)")
ax.legend(fontsize=8)
gap = non_member_losses.mean() - member_losses.mean()
ax.text(0.97, 0.97, f"Gap = {gap:.3f}", transform=ax.transAxes,
        ha="right", va="top", fontsize=10, color="#333",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray"))

# Box plot
ax = axes[1]
bp = ax.boxplot([member_losses, non_member_losses],
                labels=["Members\n(train)", "Non-members\n(test)"],
                patch_artist=True, notch=False,
                medianprops=dict(color="white", linewidth=2))
bp["boxes"][0].set_facecolor("#e07b54")
bp["boxes"][1].set_facecolor("#5b8db8")
for el in bp["whiskers"] + bp["caps"] + bp["fliers"]:
    el.set(color="#555", linewidth=1)
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Loss Spread (box plot)\n(overlapping boxes = good privacy)")
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("loss_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → loss_distribution.png")

In [ ]:
# ── ROC curve for MIA ─────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, roc_auc_score

labels_arr = np.concatenate([np.ones(len(member_losses)), np.zeros(len(non_member_losses))])
scores_arr = np.concatenate([-member_losses, -non_member_losses])  # lower loss = more likely member

fpr, tpr, _ = roc_curve(labels_arr, scores_arr)
auc_val     = roc_auc_score(labels_arr, scores_arr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color="#e07b54", lw=2, label=f"MIA ROC (AUC = {auc_val:.3f})")
ax.plot([0, 1], [0, 1], color="green", linestyle="--", lw=1.5, label="Random guess (AUC = 0.5, ideal)")
ax.fill_between(fpr, tpr, alpha=0.08, color="#e07b54")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Membership Inference Attack — ROC Curve\n(closer to diagonal = better privacy)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("mia_roc.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"MIA AUC = {auc_val:.4f}  |  Saved → mia_roc.png")

In [ ]:
# ── Adaptive noise scale: how σ grows with contribution count ─────────────────
import math

counts = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500])
sigma_uniform  = np.ones(len(counts)) * BASE_SIGMA
sigma_adaptive = BASE_SIGMA * np.log1p(counts)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(counts, sigma_adaptive, color="#4caf7d", lw=2.5, marker="o", markersize=6,
        label=f"Adaptive σ = {BASE_SIGMA} × log(1 + count)")
ax.axhline(BASE_SIGMA, color="#e07b54", linestyle="--", lw=1.8,
           label=f"Uniform σ = {BASE_SIGMA} (standard Subject-DP)")

# Annotate a few points
for c, s in zip(counts, sigma_adaptive):
    ax.annotate(f"σ={s:.2f}", (c, s), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=7.5, color="#2d6e4e")

ax.set_xscale("log")
ax.set_xticks(counts)
ax.set_xticklabels(counts)
ax.set_xlabel("Number of records contributed by a subject")
ax.set_ylabel("Noise scale σ")
ax.set_title("Adaptive Noise Scaling (Equation 3)\nHeavier contributors receive more noise")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("adaptive_noise_scale.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → adaptive_noise_scale.png")